# Notebook complet du modele H2S

Ce notebook documente le modele utilise par le systeme de surveillance de gaz H2S : preparation des donnees, classification Random Forest, prediction temporelle LSTM, tests d'inference et lien avec le serveur Flask.

## Objectif

Produire un support complet et reexecutable pour expliquer le modele du projet :

- le prototype embarque utilise un seul capteur de gaz MQ-136 pour mesurer le H2S ;
- la variable principale du modele est `C_H2S_fusion_ppm` ; dans le prototype actuel, elle est egale e `h2s_ppm` ;
- `co_ppm` n'est pas utilise dans ce notebook, meme si une ancienne base de donnees locale contient encore cette colonne ;
- Random Forest classe le niveau de danger ;
- LSTM predit l'evolution future de la concentration H2S.


## Configuration

Les parametres ci-dessous centralisent les chemins, seuils et options d'execution. Par securite, le notebook ne reentraine pas et n'ecrase pas les modeles par defaut. Pour relancer un entrainement complet, passer `RUN_TRAINING = True`, puis `SAVE_MODELS = True` seulement apres validation des resultats.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

# Recherche robuste de la racine du projet, que le notebook soit lance depuis /notebooks ou depuis la racine.
def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path(r"C:\Users\labul\gas_monitoring_system\gas_monitoring_system")]
    for candidate in candidates:
        if (candidate / "models").exists() and (candidate / "server").exists():
            return candidate.resolve()
    raise FileNotFoundError("Impossible de trouver la racine du projet gas_monitoring_system.")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "gas_data.csv"
LEGACY_DATASET_PATH = PROJECT_ROOT / "notebooks" / "DATASET01.csv"
MODELS_DIR = PROJECT_ROOT / "models"
RF_MODEL_PATH = MODELS_DIR / "random_forest_h2s_fusion.pkl"
LSTM_MODEL_PATH = MODELS_DIR / "lstm_h2s_fusion.h5"
METADATA_PATH = MODELS_DIR / "metadata_h2s_fusion.json"

# Options de securite.
RUN_TRAINING = False       # True pour reentrainer dans le notebook.
SAVE_MODELS = False        # True seulement si les resultats sont valides.
QUICK_MODE = True          # Limite certains calculs pour garder le notebook rapide.

# Parametres metier H2S.
H2S_NORMAL_MAX = 10.0
H2S_MODERATE_MAX = 20.0
CLASS_LABELS = {0: "NORMAL", 1: "MODERE", 2: "DANGEREUX"}
MODEL_FEATURES = ["C_H2S_fusion_ppm"]

# Parametres LSTM deployes.
LSTM_WINDOW = 60
LSTM_HORIZON = 50
EPOCHS = 8 if QUICK_MODE else 30
MAX_RF_ROWS = 60_000
MAX_LSTM_EVAL_SEQUENCES = 800

print("Racine projet :", PROJECT_ROOT)
print("Donnees principales :", DATA_PATH)
print("Modele Random Forest :", RF_MODEL_PATH)
print("Modele LSTM :", LSTM_MODEL_PATH)
print("Metadonnees :", METADATA_PATH)


### Hypotheses importantes

Le modele actuellement deploye dans le serveur utilise une logique heritee du jeu de donnees initial, ou plusieurs mesures H2S pouvaient etre fusionnees. Pour le prototype reel, il n'y a qu'un seul capteur MQ-136 : la fusion revient donc e utiliser directement `h2s_ppm`.


In [ ]:
assert MODEL_FEATURES == ["C_H2S_fusion_ppm"]
assert "co_ppm" not in MODEL_FEATURES

metadata = {}
if METADATA_PATH.exists():
    with open(METADATA_PATH, "r", encoding="utf-8") as handle:
        metadata = json.load(handle)

summary_metadata = {
    "rows_clean_metadata": metadata.get("rows_clean"),
    "system_vector": metadata.get("system_vector"),
    "rf_feature_importance": metadata.get("rf", {}).get("feature_importance"),
    "lstm_window": metadata.get("lstm", {}).get("window"),
    "lstm_horizon": metadata.get("lstm", {}).get("horizon"),
    "lstm_test_mae_ppm": metadata.get("lstm", {}).get("test_mae_ppm"),
    "lstm_test_rmse_ppm": metadata.get("lstm", {}).get("test_rmse_ppm"),
}
pd.Series(summary_metadata, name="metadata_deployee")


## etape 1 - Charger les donnees

Le notebook accepte deux formats :

- `notebooks/DATASET01.csv` si l'ancien jeu de donnees complet e plusieurs colonnes capteurs est disponible ;
- `data/gas_data.csv` pour le jeu de donnees local du projet.

Dans les deux cas, la sortie de preparation conserve une seule variable modele : `C_H2S_fusion_ppm`.


In [ ]:
def _first_existing_column(frame: pd.DataFrame, candidates: list[str]) -> str | None:
    normalized = {column.lower().strip(): column for column in frame.columns}
    for candidate in candidates:
        match = normalized.get(candidate.lower().strip())
        if match is not None:
            return match
    return None


def load_raw_dataset() -> tuple[pd.DataFrame, str, Path]:
    """Charge le meilleur jeu de donnees disponible sans utiliser la colonne CO."""
    if LEGACY_DATASET_PATH.exists():
        frame = pd.read_csv(LEGACY_DATASET_PATH)
        return frame, "legacy_dataset", LEGACY_DATASET_PATH
    if DATA_PATH.exists():
        frame = pd.read_csv(DATA_PATH)
        return frame, "project_gas_data", DATA_PATH
    raise FileNotFoundError("Aucun fichier de donnees trouve. Verifier data/gas_data.csv ou notebooks/DATASET01.csv.")

raw_data, source_kind, source_path = load_raw_dataset()
print(f"Source utilisee : {source_kind}")
print(f"Chemin : {source_path}")
print(f"Dimensions brutes : {raw_data.shape[0]:,} lignes x {raw_data.shape[1]} colonnes")
print("Colonnes :", list(raw_data.columns))
raw_data.head()


## etape 2 - Preparer le jeu H2S uniquement

Cette preparation applique les regles suivantes :

- suppression de la colonne CO si elle existe ;
- conversion de la mesure H2S en numerique ;
- creation de `C_H2S_fusion_ppm` ;
- correction simple des valeurs manquantes de temperature/humidite par mediane ;
- creation des classes metier e partir des seuils H2S.


In [ ]:
def build_h2s_frame(raw_frame: pd.DataFrame) -> pd.DataFrame:
    frame = raw_frame.copy()
    frame.columns = [str(column).strip() for column in frame.columns]

    # Ancien jeu e plusieurs capteurs H2S : on fusionne les colonnes disponibles.
    legacy_sensor_columns = [
        column for column in frame.columns
        if column.lower().replace(" ", "") in {
            "sensor1[ppm]", "sensor2[ppm]", "sensor3[ppm]", "sensor4[ppm]",
            "sensor1", "sensor2", "sensor3", "sensor4"
        }
    ]

    h2s_column = _first_existing_column(frame, ["h2s_ppm", "H2S", "H2S_ppm", "C_H2S_fusion_ppm"])
    if legacy_sensor_columns:
        for column in legacy_sensor_columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame["C_H2S_fusion_ppm"] = frame[legacy_sensor_columns].mean(axis=1)
        data_mode = "fusion_legacy_multi_capteurs"
    elif h2s_column is not None:
        frame[h2s_column] = pd.to_numeric(frame[h2s_column], errors="coerce")
        frame["C_H2S_fusion_ppm"] = frame[h2s_column]
        data_mode = "capteur_unique_mq136"
    else:
        raise ValueError("Aucune colonne H2S exploitable trouvee.")

    # Colonnes environnementales, si disponibles.
    temperature_column = _first_existing_column(frame, ["temperature", "Temperature", "Temperature[C]", "T", "T[degC]"])
    humidity_column = _first_existing_column(frame, ["humidity", "Humidity", "Humidity[%]", "H", "RH"])
    timestamp_column = _first_existing_column(frame, ["timestamp", "datetime", "date", "time", "DateTime"])

    prepared = pd.DataFrame()
    prepared["timestamp"] = frame[timestamp_column] if timestamp_column else np.arange(len(frame))
    prepared["C_H2S_fusion_ppm"] = pd.to_numeric(frame["C_H2S_fusion_ppm"], errors="coerce")
    prepared["temperature"] = pd.to_numeric(frame[temperature_column], errors="coerce") if temperature_column else np.nan
    prepared["humidity"] = pd.to_numeric(frame[humidity_column], errors="coerce") if humidity_column else np.nan

    prepared["source_mode"] = data_mode
    prepared["co_ignored"] = "co_ppm" in [column.lower() for column in frame.columns]

    # Nettoyage physique simple.
    prepared = prepared.dropna(subset=["C_H2S_fusion_ppm"]).copy()
    prepared = prepared[prepared["C_H2S_fusion_ppm"] >= 0].copy()

    for column in ["temperature", "humidity"]:
        if prepared[column].isna().all():
            prepared[column] = 0.0
        else:
            prepared[column] = prepared[column].fillna(prepared[column].median())

    prepared["risk_class"] = np.select(
        [
            prepared["C_H2S_fusion_ppm"] < H2S_NORMAL_MAX,
            prepared["C_H2S_fusion_ppm"] < H2S_MODERATE_MAX,
        ],
        [0, 1],
        default=2,
    ).astype(int)
    prepared["risk_label"] = prepared["risk_class"].map(CLASS_LABELS)

    prepared["timestamp_dt"] = pd.to_datetime(prepared["timestamp"], errors="coerce")
    if prepared["timestamp_dt"].notna().sum() == 0:
        prepared["timestamp_dt"] = pd.date_range("2026-01-01", periods=len(prepared), freq="s")

    return prepared.reset_index(drop=True)

h2s_data = build_h2s_frame(raw_data)
print(f"Donnees preparees : {h2s_data.shape[0]:,} lignes")
print("Mode source :", h2s_data["source_mode"].iloc[0])
print("Colonne CO presente mais ignoree :", bool(h2s_data["co_ignored"].any()))
h2s_data.head()


## etape 3 - Contreles rapides de qualite

Ces contreles verifient que les donnees preparees correspondent bien au modele H2S uniquement.


In [ ]:
quality_report = pd.DataFrame({
    "controle": [
        "lignes_pretes",
        "valeurs_h2s_manquantes",
        "h2s_negatif",
        "temperature_manquante",
        "humidite_manquante",
        "classes_invalides",
        "co_utilise_par_le_modele",
    ],
    "valeur": [
        len(h2s_data),
        int(h2s_data["C_H2S_fusion_ppm"].isna().sum()),
        int((h2s_data["C_H2S_fusion_ppm"] < 0).sum()),
        int(h2s_data["temperature"].isna().sum()),
        int(h2s_data["humidity"].isna().sum()),
        int(~h2s_data["risk_class"].isin([0, 1, 2]).sum()) if False else int((~h2s_data["risk_class"].isin([0, 1, 2])).sum()),
        "non",
    ]
})
quality_report


In [ ]:
class_distribution = (
    h2s_data["risk_label"]
    .value_counts()
    .rename_axis("classe")
    .reset_index(name="nombre")
)
class_distribution["pourcentage"] = (100 * class_distribution["nombre"] / class_distribution["nombre"].sum()).round(2)
class_distribution


In [ ]:
h2s_data[["C_H2S_fusion_ppm", "temperature", "humidity"]].describe().round(3)


## etape 4 - Visualiser la concentration H2S

Les graphiques ci-dessous servent e verifier rapidement l'amplitude des mesures et la presence des zones de danger.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(h2s_data["C_H2S_fusion_ppm"], bins=40, color="#2f80ed", edgecolor="white")
axes[0].axvline(H2S_NORMAL_MAX, color="#f2994a", linestyle="--", label="Seuil modere 10 ppm")
axes[0].axvline(H2S_MODERATE_MAX, color="#eb5757", linestyle="--", label="Seuil dangereux 20 ppm")
axes[0].set_title("Distribution des mesures H2S")
axes[0].set_xlabel("H2S (ppm)")
axes[0].set_ylabel("Nombre de mesures")
axes[0].legend()

plot_sample = h2s_data.sort_values("timestamp_dt").tail(min(600, len(h2s_data)))
axes[1].plot(plot_sample["timestamp_dt"], plot_sample["C_H2S_fusion_ppm"], color="#27ae60", linewidth=1.5)
axes[1].axhline(H2S_NORMAL_MAX, color="#f2994a", linestyle="--")
axes[1].axhline(H2S_MODERATE_MAX, color="#eb5757", linestyle="--")
axes[1].set_title("Dernieres mesures H2S")
axes[1].set_xlabel("Temps")
axes[1].set_ylabel("H2S (ppm)")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


## etape 5 - Modele Random Forest

Random Forest recoit uniquement `C_H2S_fusion_ppm` et renvoie une classe de danger :

- `0` : normal ;
- `1` : modere ;
- `2` : dangereux.

Comme les classes sont construites e partir de seuils H2S, le modele apprend principalement la frontiere entre les seuils 10 ppm et 20 ppm.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
import joblib

rf_frame = h2s_data[MODEL_FEATURES + ["risk_class"]].dropna().copy()

if QUICK_MODE and len(rf_frame) > MAX_RF_ROWS:
    # echantillonnage deterministe par classe pour garder le notebook leger.
    per_class = max(1000, MAX_RF_ROWS // max(1, rf_frame["risk_class"].nunique()))
    rf_frame = (
        rf_frame.groupby("risk_class", group_keys=False)
        .apply(lambda group: group.sample(min(len(group), per_class), random_state=42))
        .reset_index(drop=True)
    )

X = rf_frame[MODEL_FEATURES]
y = rf_frame["risk_class"].astype(int)
stratify_target = y if y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=stratify_target
)

if RUN_TRAINING or not RF_MODEL_PATH.exists():
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    )
    rf_model.fit(X_train, y_train)
    rf_origin = "modele_reentraine_dans_notebook"
else:
    rf_model = joblib.load(RF_MODEL_PATH)
    rf_origin = "modele_deploye_charge_depuis_models"

rf_predictions = rf_model.predict(X_test)
rf_metrics = {
    "origine": rf_origin,
    "n_lignes_rf": len(rf_frame),
    "accuracy": accuracy_score(y_test, rf_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, rf_predictions),
    "f1_macro": f1_score(y_test, rf_predictions, average="macro"),
    "features": MODEL_FEATURES,
}
pd.Series(rf_metrics)


In [ ]:
report = classification_report(
    y_test,
    rf_predictions,
    labels=[0, 1, 2],
    target_names=[CLASS_LABELS[i] for i in [0, 1, 2]],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T.round(4)
report_df


In [ ]:
cm = confusion_matrix(y_test, rf_predictions, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm, index=[f"reel_{CLASS_LABELS[i]}" for i in [0, 1, 2]], columns=[f"pred_{CLASS_LABELS[i]}" for i in [0, 1, 2]])
cm_df


In [ ]:
if hasattr(rf_model, "feature_importances_"):
    feature_importance = pd.DataFrame({
        "feature": MODEL_FEATURES,
        "importance": rf_model.feature_importances_,
    }).sort_values("importance", ascending=False)
else:
    feature_importance = pd.DataFrame({"feature": MODEL_FEATURES, "importance": [np.nan]})
feature_importance


## etape 6 - Preparer les sequences LSTM

Le LSTM utilise une fenetre temporelle de 60 mesures pour predire la concentration H2S future e l'horizon de 50 pas. Cette section prepare les sequences au format `(nombre_sequences, 60, 1)`.


In [ ]:
def get_lstm_scaler_bounds(series: np.ndarray) -> tuple[float, float]:
    meta_lstm = metadata.get("lstm", {})
    min_value = meta_lstm.get("h2s_min_train")
    max_value = meta_lstm.get("h2s_max_train")
    if min_value is None or max_value is None or max_value <= min_value:
        min_value = float(np.nanmin(series))
        max_value = float(np.nanmax(series))
    if max_value <= min_value:
        max_value = min_value + 1.0
    return float(min_value), float(max_value)


def normalize_h2s(values: np.ndarray, min_value: float, max_value: float) -> np.ndarray:
    return np.clip((values - min_value) / (max_value - min_value), 0, 1)


def denormalize_h2s(values: np.ndarray, min_value: float, max_value: float) -> np.ndarray:
    return values * (max_value - min_value) + min_value


def make_lstm_sequences(normalized_series: np.ndarray, window: int, horizon: int) -> tuple[np.ndarray, np.ndarray]:
    X_sequences, y_targets = [], []
    last_start = len(normalized_series) - window - horizon + 1
    for start in range(max(0, last_start)):
        end = start + window
        target_index = end + horizon - 1
        X_sequences.append(normalized_series[start:end])
        y_targets.append(normalized_series[target_index])
    X_array = np.asarray(X_sequences, dtype=np.float32).reshape(-1, window, 1)
    y_array = np.asarray(y_targets, dtype=np.float32).reshape(-1, 1)
    return X_array, y_array

time_series_frame = h2s_data.sort_values("timestamp_dt").reset_index(drop=True)
h2s_series = time_series_frame["C_H2S_fusion_ppm"].astype(float).to_numpy()
train_min, train_max = get_lstm_scaler_bounds(h2s_series)
normalized_series = normalize_h2s(h2s_series, train_min, train_max)
X_lstm, y_lstm = make_lstm_sequences(normalized_series, LSTM_WINDOW, LSTM_HORIZON)

print("Borne min normalisation :", train_min)
print("Borne max normalisation :", train_max)
print("Forme X_lstm :", X_lstm.shape)
print("Forme y_lstm :", y_lstm.shape)


## etape 7 - Modele LSTM

La cellule suivante charge le LSTM deploye. Si `RUN_TRAINING=True`, elle cree et entraine un nouveau modele LSTM leger depuis les sequences preparees.


In [ ]:
lstm_model = None
lstm_origin = "non_charge"
lstm_history = None

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping

    if RUN_TRAINING or not LSTM_MODEL_PATH.exists():
        if len(X_lstm) < 100:
            raise ValueError("Pas assez de sequences pour entrainer le LSTM.")

        split_train = int(len(X_lstm) * 0.70)
        split_validation = int(len(X_lstm) * 0.85)
        X_train_lstm, y_train_lstm = X_lstm[:split_train], y_lstm[:split_train]
        X_validation_lstm, y_validation_lstm = X_lstm[split_train:split_validation], y_lstm[split_train:split_validation]

        lstm_model = Sequential([
            LSTM(32, input_shape=(LSTM_WINDOW, 1), return_sequences=False),
            Dropout(0.15),
            Dense(16, activation="relu"),
            Dense(1, activation="linear"),
        ])
        lstm_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
        lstm_history = lstm_model.fit(
            X_train_lstm,
            y_train_lstm,
            validation_data=(X_validation_lstm, y_validation_lstm),
            epochs=EPOCHS,
            batch_size=64,
            verbose=1,
            callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
        )
        lstm_origin = "modele_reentraine_dans_notebook"
    else:
        lstm_model = tf.keras.models.load_model(LSTM_MODEL_PATH, compile=False)
        lstm_origin = "modele_deploye_charge_depuis_models"
except Exception as exc:
    lstm_origin = f"indisponible: {type(exc).__name__}: {exc}"

print("Origine LSTM :", lstm_origin)
if lstm_model is not None:
    lstm_model.summary()


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lstm_evaluation = None
if lstm_model is not None and len(X_lstm) > 0:
    eval_start = int(len(X_lstm) * 0.80)
    eval_end = min(len(X_lstm), eval_start + MAX_LSTM_EVAL_SEQUENCES)
    X_eval_lstm = X_lstm[eval_start:eval_end]
    y_eval_lstm = y_lstm[eval_start:eval_end].reshape(-1)

    pred_norm = lstm_model.predict(X_eval_lstm, verbose=0).reshape(-1)
    y_eval_ppm = denormalize_h2s(y_eval_lstm, train_min, train_max)
    pred_ppm = denormalize_h2s(pred_norm, train_min, train_max)

    lstm_evaluation = {
        "origine": lstm_origin,
        "sequences_evaluees": len(X_eval_lstm),
        "mae_ppm": mean_absolute_error(y_eval_ppm, pred_ppm),
        "rmse_ppm": float(np.sqrt(mean_squared_error(y_eval_ppm, pred_ppm))),
        "r2": r2_score(y_eval_ppm, pred_ppm) if len(y_eval_ppm) > 1 else np.nan,
    }
    lstm_eval_frame = pd.DataFrame({
        "h2s_reel_ppm": y_eval_ppm,
        "h2s_predit_ppm": pred_ppm,
        "erreur_ppm": pred_ppm - y_eval_ppm,
    })
else:
    lstm_evaluation = {"origine": lstm_origin, "sequences_evaluees": 0, "mae_ppm": np.nan, "rmse_ppm": np.nan, "r2": np.nan}
    lstm_eval_frame = pd.DataFrame()

pd.Series(lstm_evaluation)


In [ ]:
if not lstm_eval_frame.empty:
    sample_plot = lstm_eval_frame.head(min(200, len(lstm_eval_frame)))
    plt.figure(figsize=(12, 4))
    plt.plot(sample_plot.index, sample_plot["h2s_reel_ppm"], label="H2S reel", color="#2f80ed")
    plt.plot(sample_plot.index, sample_plot["h2s_predit_ppm"], label="H2S predit", color="#eb5757", linestyle="--")
    plt.title("Comparaison LSTM : H2S reel vs H2S predit")
    plt.xlabel("Sequence d'evaluation")
    plt.ylabel("H2S (ppm)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Aucun graphique LSTM : modele indisponible ou sequences insuffisantes.")


## etape 8 - Test d'inference comme dans le serveur

Cette partie montre comment le serveur Flask utilise les modeles : une mesure H2S arrive depuis l'ESP32, le serveur cree `C_H2S_fusion_ppm`, classe le danger avec Random Forest et peut estimer une tendance future avec LSTM.


In [ ]:
def threshold_label(h2s_ppm: float) -> tuple[int, str]:
    if h2s_ppm < H2S_NORMAL_MAX:
        return 0, CLASS_LABELS[0]
    if h2s_ppm < H2S_MODERATE_MAX:
        return 1, CLASS_LABELS[1]
    return 2, CLASS_LABELS[2]


def predict_rf_for_values(values: list[float]) -> pd.DataFrame:
    inference_frame = pd.DataFrame({"C_H2S_fusion_ppm": values})
    predictions = rf_model.predict(inference_frame).astype(int)
    probabilities = rf_model.predict_proba(inference_frame) if hasattr(rf_model, "predict_proba") else np.full((len(values), 3), np.nan)

    rows = []
    for index, value in enumerate(values):
        threshold_class, threshold_name = threshold_label(value)
        predicted_class = int(predictions[index])
        rows.append({
            "h2s_ppm": value,
            "feature_envoyee_au_modele": value,
            "classe_seuil": threshold_class,
            "label_seuil": threshold_name,
            "classe_random_forest": predicted_class,
            "label_random_forest": CLASS_LABELS.get(predicted_class, "INCONNU"),
            "proba_NORMAL": probabilities[index][0] if probabilities.shape[1] > 0 else np.nan,
            "proba_MODERE": probabilities[index][1] if probabilities.shape[1] > 1 else np.nan,
            "proba_DANGEREUX": probabilities[index][2] if probabilities.shape[1] > 2 else np.nan,
        })
    return pd.DataFrame(rows)

predict_rf_for_values([2.0, 8.5, 12.0, 19.5, 25.0, 31.0]).round(4)


In [ ]:
if lstm_model is not None and len(h2s_series) >= LSTM_WINDOW:
    last_window_ppm = h2s_series[-LSTM_WINDOW:]
    last_window_norm = normalize_h2s(last_window_ppm, train_min, train_max).reshape(1, LSTM_WINDOW, 1)
    future_norm = float(lstm_model.predict(last_window_norm, verbose=0).reshape(-1)[0])
    future_h2s = float(denormalize_h2s(np.array([future_norm]), train_min, train_max)[0])
    future_class, future_label = threshold_label(future_h2s)

    lstm_inference = pd.DataFrame([{
        "fenetre_mesures": LSTM_WINDOW,
        "horizon_pas": LSTM_HORIZON,
        "dernier_h2s_ppm": float(last_window_ppm[-1]),
        "prediction_future_h2s_ppm": future_h2s,
        "classe_future": future_class,
        "label_future": future_label,
    }])
else:
    lstm_inference = pd.DataFrame([{
        "fenetre_mesures": LSTM_WINDOW,
        "horizon_pas": LSTM_HORIZON,
        "dernier_h2s_ppm": np.nan,
        "prediction_future_h2s_ppm": np.nan,
        "classe_future": np.nan,
        "label_future": "LSTM indisponible",
    }])

lstm_inference.round(4)


## etape 9 - Sauvegarde contrelee des modeles

Cette cellule evite d'ecraser les modeles du serveur par accident. Elle ne sauvegarde que si `RUN_TRAINING=True` et `SAVE_MODELS=True`.


In [ ]:
new_metadata = {
    "project": "gas_monitoring_system",
    "gas_sensor": "MQ-136",
    "gas_monitored": "H2S",
    "co_used": False,
    "main_training_variable": "C_H2S_fusion_ppm",
    "deployment_mapping": "C_H2S_fusion_ppm = h2s_ppm pour le prototype e capteur unique",
    "thresholds_ppm": {
        "NORMAL": f"H2S < {H2S_NORMAL_MAX}",
        "MODERE": f"{H2S_NORMAL_MAX} <= H2S < {H2S_MODERATE_MAX}",
        "DANGEREUX": f"H2S >= {H2S_MODERATE_MAX}",
    },
    "rf_metrics_notebook": rf_metrics,
    "lstm_metrics_notebook": lstm_evaluation,
    "lstm_window": LSTM_WINDOW,
    "lstm_horizon": LSTM_HORIZON,
    "source_path": str(source_path),
}

if RUN_TRAINING and SAVE_MODELS:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(rf_model, RF_MODEL_PATH)
    if lstm_model is not None:
        lstm_model.save(LSTM_MODEL_PATH)
    with open(METADATA_PATH, "w", encoding="utf-8") as handle:
        json.dump(new_metadata, handle, ensure_ascii=False, indent=2)
    print("Modeles sauvegardes dans", MODELS_DIR)
else:
    print("Sauvegarde desactivee : aucun modele existant n'a ete ecrase.")

pd.Series({
    "RUN_TRAINING": RUN_TRAINING,
    "SAVE_MODELS": SAVE_MODELS,
    "rf_path": str(RF_MODEL_PATH),
    "lstm_path": str(LSTM_MODEL_PATH),
    "metadata_path": str(METADATA_PATH),
})


## Contreles finaux

Ces assertions verifient que le notebook reste coherent avec la version actuelle du projet : un seul capteur de gaz H2S, pas de CO dans les variables modele, et des fichiers modele disponibles.


In [ ]:
assert "co_ppm" not in MODEL_FEATURES, "La colonne CO ne doit pas entrer dans le modele."
assert MODEL_FEATURES == ["C_H2S_fusion_ppm"], "Le modele RF doit rester H2S uniquement."
assert h2s_data["C_H2S_fusion_ppm"].notna().all(), "Toutes les mesures H2S preparees doivent etre renseignees."
assert RF_MODEL_PATH.exists() or RUN_TRAINING, "Le modele Random Forest doit exister ou etre reentraene."

checks = pd.DataFrame({
    "controle": [
        "capteur_gaz_unique",
        "co_exclu_du_modele",
        "feature_rf_unique",
        "rf_disponible",
        "lstm_disponible_ou_optionnel",
    ],
    "statut": [
        "OK - MQ-136 / H2S",
        "OK - co_ppm ignore",
        "OK - C_H2S_fusion_ppm",
        "OK" if RF_MODEL_PATH.exists() or RUN_TRAINING else "A verifier",
        "OK" if LSTM_MODEL_PATH.exists() else "Optionnel / e reentrainer",
    ],
})
checks


## Suite

Pour utiliser ce notebook dans le rapport ou la soutenance :

1. executer toutes les cellules dans l'ordre ;
2. garder `RUN_TRAINING=False` pour demontrer les modeles deja deployes ;
3. passer `RUN_TRAINING=True` uniquement pour refaire l'experience d'entrainement ;
4. ne passer `SAVE_MODELS=True` qu'apres validation des metriques ;
5. expliquer que `C_H2S_fusion_ppm = h2s_ppm` dans la version actuelle du prototype, car il n'y a qu'un seul capteur MQ-136.
